In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import datetime
import scipy.stats
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.colors as cl
import matplotlib.dates as mdates
import matplotlib
import statsmodels.api as sm
from statsmodels.regression.rolling import RollingOLS
import seaborn as sns
from scipy.optimize import curve_fit
from scipy import signal 
from matplotlib.patches import Patch
from matplotlib.lines import Line2D


In [ ]:
def decimal_year_to_datetime(decimal_year):
    year = decimal_year.astype(int)
    decimal_part = decimal_year - year
    start_of_year = pd.to_datetime([f'{y}-01-01' for y in year])
    seconds_in_year = (pd.to_datetime([f'{y+1}-01-01' for y in year]) - start_of_year).total_seconds()
    seconds = decimal_part * seconds_in_year
    return start_of_year + pd.to_timedelta(seconds, unit='s')

def datetime_to_decimal_year(datetime_):
    year = datetime_.year
    days = (datetime_.date() - datetime.date(year, 1,1)).days
    res = year + days/365.25
    return res

def sine_function(x, A, B, C, D):
    return A * np.sin(B * x + C) + D


def sin_fitter(x_data, y_data):

    #### Initial guesses
    guess_amplitude = (np.max(y_data) - np.min(y_data))/2
    guess_frequency =  2*np.pi # A reasonable starting point
    guess_phase = 0
    guess_offset = np.mean(y_data)
    p0 = [guess_amplitude, guess_frequency, guess_phase, guess_offset]

    params, covariance = curve_fit(sine_function, x_data, y_data, p0=p0)
    sin_fit = sine_function(x_data, *params)
    
    return x_data, sin_fit

def FFT_res(x, si):
    n = len(x)
    x_FFT_shifted = np.fft.fftshift(np.fft.fft(x))
    x_FFT_freq_shifted = np.fft.fftshift(np.fft.fftfreq(n, d=si))
    return x_FFT_freq_shifted, x_FFT_shifted


def fft_noramlize(fft_res):
    n = len(fft_res)
    norm_fft = np.abs(fft_res)/(np.max(np.abs(fft_res)))
    return norm_fft

def line_fit(X,Y):
    P = np.polyfit(X,Y,1)
    dx = np.max(X) - np.min(X)
    X_new = np.array([np.min(X)-dx*0.05, np.max(X)+dx*0.05])
    Y_new = np.polyval(P, X_new)
    return X_new, Y_new, P

In [ ]:
window_size = 3
window_size2 = 36


df = pd.read_excel('../age_model/interp_20251220.xlsx')
paths = list(df['path_name'].value_counts().index.values)
for path in paths:
    df.loc[df['path_name']==path, "path_ma"] = df.loc[df['path_name']==path, 'd13c'].rolling(window_size, center=True).mean()
#df[['d13c_ma','d18o_ma']] = df[['d13c','d18o']].rolling(window_size, center=True).mean()
df_sorted = df.sort_values("age_absolute")
df_sorted['age_ws'] = df_sorted['age_absolute'].rolling(window_size, center=True).mean()
df_sorted['age_ws2'] = df_sorted['age_absolute'].rolling(window_size2, center=True).mean()

df_sorted[['d13c_ma','d18o_ma']] = df_sorted[['d13c','d18o']].rolling(window_size, center=True).mean()
df_sorted[['d13c_ma2','d18o_ma2']] = df_sorted[['d13c','d18o']].rolling(window_size2, center=True).mean()
df_sorted[['d13c_std','d18o_std']] = df_sorted[['d13c','d18o']].rolling(window_size2, center=True).std()
df_sorted['d13c_no_tectonic'] = df_sorted['d13c'] - df_sorted['d13c_ma2'].fillna(0)
df_sorted['d18o_no_tectonic'] = df_sorted['d18o'] - df_sorted['d18o_ma2'].fillna(0)
df_sorted[['d13c_notec_ma','d18o_notec_ma']] = df_sorted[['d13c_no_tectonic','d18o_no_tectonic']].rolling(window_size, center=True).mean()

#-----------------------RESAMPLE MEAN & STD-----------------------------------------------------

df_sorted['age_dt'] = decimal_year_to_datetime(df_sorted['age_absolute'])
df_resampled_ = df_sorted.set_index('age_dt')
df_resampled_.drop(['Name', 'Mass', 'Amp 44', 'Area 44', 'index','sample','age1','path_name',
       'path_num', 'path_ma', 'age_ws', 'age_ws2',
       'd13c_ma', 'd18o_ma', 'd13c_ma2', 'd18o_ma2', 'd13c_std', 'd18o_std'], axis=1, inplace=True)
df_resampled = df_resampled_.resample('30D').mean()

df_resampled.interpolate(inplace=True)
df_resampled[['d13c_ma','d18o_ma']] = df_resampled[['d13c','d18o']].rolling(window_size, center=True).mean()
df_resampled[['d13c_ma2','d18o_ma2']] = df_resampled[['d13c','d18o']].rolling(window_size2, center=True).mean()
df_resampled[['d13c_std','d18o_std']] = df_resampled[['d13c','d18o']].rolling(window_size2, center=True).std()

#-----------------------------------------------------------------------------------------------


#-----------------------CENTRAL DIFF METHOD-----------------------------------------------------
window = 12
col = 'd18o'

last = df_resampled.rolling(window, center=True).agg(lambda x: x.iloc[-1])
first = df_resampled.rolling(window, center=True).agg(lambda x: x.iloc[0])
df_resampled[col+'_cdm'] = (last[col] - first[col]) / (last['age_absolute'] - first['age_absolute'])

window = 12
col = 'd13c'

last = df_resampled.rolling(window, center=True).agg(lambda x: x.iloc[-1])
first = df_resampled.rolling(window, center=True).agg(lambda x: x.iloc[0])
df_resampled[col+'_cdm'] = (last[col] - first[col]) / (last['age_absolute'] - first['age_absolute'])
#-----------------------------------------------------------------------------------------------


#-----------------------------Moving OSL---------------------------------------
coefs_centered = pd.DataFrame()

# Define multiple windows and target columns
windows = [36, 36]
cols = ['d18o', 'd13c']

for window, col in zip(windows, cols):
    
    # Prepare data (assumes df_resampled contains all variables)
    data = df_resampled[[col, 'age_absolute']].dropna()

    if len(data) < window:
        print(f"Not enough data for column '{col}' with window {window}. Skipping.")
        continue
    
    # Setup endogenous and exogenous variables
    endog = data[col]
    exog = sm.add_constant(data['age_absolute'])

    # Fit rolling OLS
    rols = RollingOLS(endog, exog, window=window)
    rres = rols.fit()

    # Shift result to center and rename columns
    coefs = rres.params.shift(-(window // 2)).dropna()
    coefs = coefs.rename(columns={'const': f'{col}_intercept', 'age_absolute': f'{col}_slope'})

    # Merge into result
    coefs_centered = coefs_centered.join(coefs, how='outer')

# Add shared age column (assumes time index is same)
coefs_centered['age_absolute'] = df_resampled['age_absolute']

# Drop rows with NaNs in all slope columns (optional)
slope_cols = [f"{col}_slope" for col in cols]
coefs_centered.dropna(subset=slope_cols, how='all', inplace=True)
 
#-----------------------------------------------------------------------------------------------


#---------------------------------------FFT-----------------------------------------------------
d13c_fft_freq, d13c_fft = FFT_res(df_resampled['d13c'], 30)
d18o_fft_freq, d18o_fft = FFT_res(df_resampled['d18o'], 30)

df_resampled[['d13c_std','d18o_std']] = df_resampled[['d13c','d18o']].rolling(window=window_size2, center=True).std()
d13c_stds = df_resampled.dropna()['d13c_std']
d18o_stds = df_resampled.dropna()['d18o_std']
d13c_std_fft_freq, d13c_std_fft = FFT_res(d13c_stds, 30)
d18o_std_fft_freq, d18o_std_fft = FFT_res(d18o_stds, 30)
#-----------------------------------------------------------------------------------------------

#---------------------------------------DARK BAND DATES-----------------------------------------
dark_bands_end = df_sorted[~df_sorted['age_end'].isna()]['age_absolute'].drop_duplicates()
dark_bands_end
#-----------------------------------------------------------------------------------------------
df_resampled

In [ ]:
fred = pd.read_excel('../../2025-08-22_FredsData-post2008/FredData_editedMemo.xlsx').dropna()
fred['d18o_pdb'] = -(fred['d18o']-20.91)/1.03091
fred[['d13c_ma','d18o_ma']] = fred[['d13c','d18o_pdb']].rolling(window_size, center=True).mean()
fred[['d13c_std','d18o_std']] = fred[['d13c','d18o_pdb']].rolling(10, center=True).std()
fred['d13c_no_tectonic'] = fred['d13c']-fred['d13c_ma']
fred['d18o_no_tectonic'] = fred['d18o']-fred['d18o_ma']
fred_resampled = fred.copy()
fred_resampled['age_dt'] = decimal_year_to_datetime(fred_resampled['decimal_year'])
fred_resampled.set_index('age_dt', inplace=True)
fred_resampled = fred_resampled.resample('30D').mean()
fred_resampled.interpolate(inplace=True)
fred_resampled

In [ ]:
tide = pd.read_csv('../../2025-10-10_Fred-TideGauge/memo_edited.csv')
tide['age_dt'] = decimal_year_to_datetime(tide['Date'])#.dt.date
tide = tide.set_index('age_dt')
tide_resampled = tide.resample('30D').mean()
tide_resampled.interpolate(inplace=True)
tide_resampled

In [ ]:
enso_phase = pd.read_csv('../../ENSO_Phase_Classification.csv')
enso_phase.columns=['Year','Phase']
enso_phase['Date'] = decimal_year_to_datetime(enso_phase['Year'])
#enso_phase.set_index('Date', inplace=True)
enso_phase['Phase'] = enso_phase['Phase'].str.replace('Warm Phase', 'W').str.replace('Cold Phase', 'C').str.replace('Neutral Phase','N')


enso = pd.read_csv("../../ENSO_Time_Series.csv", parse_dates=[0])
#enso = enso.set_index("Date")
enso['day_of_year'] = enso['Date'].dt.dayofyear
enso['is_leap'] = enso['Date'].dt.is_leap_year
enso['total_days'] = np.where(enso['is_leap'], 366, 365)
enso['decimal_year'] = enso['Date'].dt.year + (enso['day_of_year'] - 1) / enso['total_days']


margin = 0.5
enso['phase'] = 'N'
enso.loc[enso['ENSO']>=margin,'phase'] = "C"
enso.loc[enso['ENSO']<=-margin,'phase'] = "W"
enso['phase_group'] = (enso['phase'] != enso['phase'].shift()).cumsum()
blocks = enso.groupby('phase_group').agg(
    start=('Date', 'first'),
    end=('Date', 'last'),
    value=('phase', 'first')
).reset_index(drop=True)

ENSO_match = np.array([[1982.2, 1984.1],
                       [1986.75, 1988.3],
                       [1992.1, 1992.96],
                       [1997.2, 1998.8]])

enso_fft_freq, enso_fft = FFT_res(enso['ENSO'], 30)


enso


In [ ]:
phase_color = {'W':'red', 'C':'blue','N':'white'}
#enso_phase_selected = enso_phase[(enso_phase['Year']>1925)&(enso_phase['Year']<2014)&(enso_phase['Phase'] != 'N')]

#fig = go.Figure()
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=df_sorted['age_absolute'], y=df_sorted['d13c'],  mode="markers", marker_color="lightgreen",text=df_sorted['Name'],  name='d13C'),secondary_y=False)
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_ma'], mode="lines", line_color='green', name='average'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_ma2'], mode="lines", line_color='red', name='average'), secondary_y=False)

fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c'], mode="markers", line_color='turquoise', name='fred'), secondary_y=False)
fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c_ma'], mode="lines", line_color='darkturquoise', name='average'), secondary_y=False)


#fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_std'], mode="lines", line_color='darkblue', name='std'), secondary_y=True)
#fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c_std'], mode="lines", line_color='darkblue', name='std_fred'), secondary_y=True)

fig.add_trace(go.Scatter(x=tide['Date'], y=tide['Sea Level'], mode="lines", line_color='#F54927', name='Tide Gauge', line_width=1), secondary_y=True)



for i, dates in enumerate(dark_bands_end):
    fig.add_vrect(x0=dates-0.5, 
                  x1=dates, 
                  fillcolor=phase_color['C'], opacity=0.1)



# Add figure title
fig.update_layout(width=1000, height=600, title_text="d13C and ENSO", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta C ^{13}$', secondary_y=False)
fig.update_yaxes(title_text="Standard Deviation", secondary_y=True)

fig.show()
#fig.write_html('d13c_std.html')


In [ ]:
x_data, y_data = df_sorted['age_absolute'].values, df_sorted['d13c_no_tectonic'].values

#### Initial guesses
guess_amplitude = (np.max(y_data) - np.min(y_data))/2
guess_frequency =  2*np.pi # A reasonable starting point
guess_phase = 0
guess_offset = np.mean(y_data)
p0 = [guess_amplitude, guess_frequency, guess_phase, guess_offset]

params, covariance = curve_fit(sine_function, x_data, y_data, p0=p0)
sin_fit_c = sine_function(x_data, *params)
print(params[1]/(2*np.pi), np.abs(params[2]/(2*np.pi)%2))

In [ ]:
x_data, y_data = df_sorted['age_absolute'].values, df_sorted['d18o_no_tectonic'].values

#### Initial guesses
guess_amplitude = (np.max(y_data) - np.min(y_data))/2
guess_frequency =  2*np.pi # A reasonable starting point
guess_phase = 0
guess_offset = np.mean(y_data)
p0 = [guess_amplitude, guess_frequency, guess_phase, guess_offset]

params, covariance = curve_fit(sine_function, x_data, y_data, p0=p0)
sin_fit_o = sine_function(x_data, *params)
print(params[1]/(2*np.pi), np.abs(params[2]/(2*np.pi)%2))

In [ ]:
1.7873755733746464 - 0.6998117699415278

In [ ]:
phase_color = {'W':'red', 'C':'blue','N':'white'}

fig, ax = plt.subplots(1, figsize=(8, 4))

ax.plot(x_data, sin_fit_c-sin_fit_c.mean(), label=r'$\delta^{13}$C')
ax.plot(x_data, sin_fit_o-sin_fit_o.mean(), label=r'$\delta^{18}$O')

for i, dates in enumerate(dark_bands_end):
    rect = matplotlib.patches.Rectangle(xy=(dates-0.5, -0.15), width=0.5, height=1, facecolor=phase_color['C'], alpha=0.1)
    ax.add_patch(rect)


ax.legend()
start = 1990
ax.set_xlim(start,start+10)
ax.set_ylim(-0.1,0.1)

In [ ]:
import numpy
from matplotlib import pyplot

import pycwt as wavelet
from pycwt.helpers import find

In [ ]:
dat = df_resampled.loc[:'2000','d13c_notec_ma'].dropna().values
par_name = 'd13c'
title = "d13C without tectonic signal"
label = "d13C"
units = ""
t0 = 1926.3490
dt = 1/12  # In years

N = dat.size
t = numpy.arange(0, N) * dt + t0
locals()['t_'+par_name] = t


p = numpy.polyfit(t - t0, dat, 1)
dat_notrend = dat - numpy.polyval(p, t - t0)
std = dat_notrend.std()  # Standard deviation
var = std**2  # Variance
dat_norm = dat_notrend / std  # Normalized dataset
locals()['dat_norm_'+par_name] = dat_norm

mother = wavelet.Morlet(6)
s0 = 2 * dt  # Starting scale, in this case 2 * dt years = 2 months
dj = 1 / 24  # Twelve sub-octaves per octaves
J = 10 / dj  # Seven powers of two with dj sub-octaves
alpha, _, _ = wavelet.ar1(dat)  # Lag-1 autocorrelation for red noise

wave, scales, freqs, coi, fft, fftfreqs = wavelet.cwt(dat_norm, dt, dj, s0, J, mother)
iwave = wavelet.icwt(wave, scales, dt, dj, mother) * std

power = (numpy.abs(wave)) ** 2
fft_power = numpy.abs(fft) ** 2
period = 1 / freqs
locals()['period_'+par_name] = period

power /= scales[:, None]

In [ ]:
signif, fft_theor = wavelet.significance(
    1.0, dt, scales, 0, alpha, significance_level=0.95, wavelet=mother
)
sig95 = numpy.ones([1, N]) * signif[:, None]
sig95 = power / sig95
locals()['sig95_'+par_name] = sig95


glbl_power = power.mean(axis=1)
dof = N - scales  # Correction for padding at edges
glbl_signif, tmp = wavelet.significance(
    var, dt, scales, 1, alpha, significance_level=0.95, dof=dof, wavelet=mother
)

sel=np.array([1,2])
Cdelta = mother.cdelta
scale_avg = (scales * numpy.ones((N, 1))).transpose()
scale_avg = power / scale_avg  # As in Torrence and Compo (1998) equation 24
scale_avg = var * dj * dt / Cdelta * scale_avg[sel, :].sum(axis=0)
scale_avg_signif, tmp = wavelet.significance(
    var,
    dt,
    scales,
    2,
    alpha,
    significance_level=0.95,
    dof=[scales[sel[0]], scales[sel[-1]]],
    wavelet=mother,
)

In [ ]:
# Prepare the figure
pyplot.close("all")
pyplot.ioff()
figprops = dict(figsize=(11, 8), dpi=72)
fig = pyplot.figure(**figprops)

# First sub-plot, the original time series anomaly and inverse wavelet
# transform.
ax = pyplot.axes([0.1, 0.75, 0.65, 0.2])
ax.plot(t, iwave, "-", linewidth=1, color=[0.5, 0.5, 0.5])
ax.plot(t, dat, "k", linewidth=1.5)
ax.set_title("a) {}".format(title))
ax.set_ylabel(r"{} [{}]".format(label, units))

# Second sub-plot, the normalized wavelet power spectrum and significance
# level contour lines and cone of influece hatched area. Note that period
# scale is logarithmic.
bx = pyplot.axes([0.1, 0.37, 0.65, 0.28], sharex=ax)
levels = [0.0625, 0.125, 0.25, 0.5, 1, 2, 4, 8, 16]
bx.contourf(t, numpy.log2(period), numpy.log2(power), numpy.log2(levels), extend="both", cmap=pyplot.cm.viridis)
extent = [t.min(), t.max(), 0, max(period)]
bx.contour(t, numpy.log2(period), sig95, [-99, 1], colors="k", linewidths=2, extent=extent)
bx.fill(numpy.concatenate([t, t[-1:] + dt, t[-1:] + dt, t[:1] - dt, t[:1] - dt]),
    numpy.concatenate([numpy.log2(coi), [1e-9],  numpy.log2(period[-1:]), numpy.log2(period[-1:]), [1e-9],]),"k",  alpha=0.3, hatch="x")
bx.set_title("b) {} Wavelet Power Spectrum ({})".format(label, mother.name))
bx.set_ylabel("Period (years)")
bx.hlines(numpy.log2(1), t[0], t[-1], colors='orange', linestyles='dashed')

#
Yticks = 2 ** numpy.arange(numpy.ceil(numpy.log2(period.min())), numpy.ceil(numpy.log2(period.max())))
bx.set_yticks(numpy.log2(Yticks))
bx.set_yticklabels(Yticks)

# Third sub-plot, the global wavelet and Fourier power spectra and theoretical
# noise spectra. Note that period scale is logarithmic.
cx = pyplot.axes([0.77, 0.37, 0.2, 0.28], sharey=bx)
cx.plot(glbl_signif, numpy.log2(period), "k--")
cx.plot(var * fft_theor, numpy.log2(period), "--", color="#cccccc")
cx.plot( var * fft_power, numpy.log2(1.0 / fftfreqs), "-", color="#cccccc", linewidth=1.0)
cx.plot(var * glbl_power, numpy.log2(period), "k-", linewidth=1.5)
cx.set_title("c) Global Wavelet Spectrum")
cx.set_xlabel(r"Power [({})^2]".format(units))
cx.set_xlim([0, glbl_power.max() + var])
cx.set_ylim(numpy.log2([period.min(), period.max()]))
cx.set_yticks(numpy.log2(Yticks))
cx.set_yticklabels(Yticks)
pyplot.setp(cx.get_yticklabels(), visible=False)

# Fourth sub-plot, the scale averaged wavelet spectrum.
dx = pyplot.axes([0.1, 0.07, 0.65, 0.2], sharex=ax)
dx.axhline(scale_avg_signif, color="k", linestyle="--", linewidth=1.0)
dx.plot(t, scale_avg, "k-", linewidth=1.5)
dx.set_title("d) {}--{} year scale-averaged power".format(sel[0], sel[1]))
dx.set_xlabel("Time (year)")
dx.set_ylabel(r"Average variance [{}]".format(units))
ax.set_xlim([t.min(), t.max()])

pyplot.show()
fig.savefig('wavelet'+label+'.pdf')

In [ ]:
dat = df_resampled['d18o_ma'].dropna().values-df_resampled['d18o'].mean()
par_name = 'd18o'
title = "d18O 3 Month MA"
label = "d18O"
units = ""
t0 = 1926.3490
dt = 1/12  # In years

N = dat.size
t = numpy.arange(0, N) * dt + t0
locals()['t_'+par_name] = t


p = numpy.polyfit(t - t0, dat, 1)
dat_notrend = dat - numpy.polyval(p, t - t0)
std = dat_notrend.std()  # Standard deviation
var = std**2  # Variance
dat_norm = dat_notrend / std  # Normalized dataset
locals()['dat_norm_'+par_name] = dat_norm

mother = wavelet.Morlet(6)
s0 = 2 * dt  # Starting scale, in this case 2 * dt years = 2 months
dj = 1 / 12  # Twelve sub-octaves per octaves
J = 7 / dj  # Seven powers of two with dj sub-octaves
alpha, _, _ = wavelet.ar1(dat)  # Lag-1 autocorrelation for red noise

wave, scales, freqs, coi, fft, fftfreqs = wavelet.cwt(dat_norm, dt, dj, s0, J, mother)
iwave = wavelet.icwt(wave, scales, dt, dj, mother) * std

power = (numpy.abs(wave)) ** 2
fft_power = numpy.abs(fft) ** 2
period = 1 / freqs
locals()['period_'+par_name] = period

power /= scales[:, None]

In [ ]:
signif, fft_theor = wavelet.significance(
    1.0, dt, scales, 0, alpha, significance_level=0.95, wavelet=mother
)
sig95 = numpy.ones([1, N]) * signif[:, None]
sig95 = power / sig95
locals()['sig95_'+par_name] = sig95


glbl_power = power.mean(axis=1)
dof = N - scales  # Correction for padding at edges
glbl_signif, tmp = wavelet.significance(
    var, dt, scales, 1, alpha, significance_level=0.95, dof=dof, wavelet=mother
)

sel=np.array([1,2])
Cdelta = mother.cdelta
scale_avg = (scales * numpy.ones((N, 1))).transpose()
scale_avg = power / scale_avg  # As in Torrence and Compo (1998) equation 24
scale_avg = var * dj * dt / Cdelta * scale_avg[sel, :].sum(axis=0)
scale_avg_signif, tmp = wavelet.significance(
    var,
    dt,
    scales,
    2,
    alpha,
    significance_level=0.95,
    dof=[scales[sel[0]], scales[sel[-1]]],
    wavelet=mother,
)

In [ ]:
# Prepare the figure
pyplot.close("all")
pyplot.ioff()
figprops = dict(figsize=(11, 8), dpi=72)
fig = pyplot.figure(**figprops)

# First sub-plot, the original time series anomaly and inverse wavelet
# transform.
ax = pyplot.axes([0.1, 0.75, 0.65, 0.2])
ax.plot(t, iwave, "-", linewidth=1, color=[0.5, 0.5, 0.5])
ax.plot(t, dat, "k", linewidth=1.5)
ax.set_title("a) {}".format(title))
ax.set_ylabel(r"{} [{}]".format(label, units))

# Second sub-plot, the normalized wavelet power spectrum and significance
# level contour lines and cone of influece hatched area. Note that period
# scale is logarithmic.
bx = pyplot.axes([0.1, 0.37, 0.65, 0.28], sharex=ax)
levels = [0.0625, 0.125, 0.25, 0.5, 1, 2, 4, 8, 16]
bx.contourf(t, numpy.log2(period), numpy.log2(power), numpy.log2(levels), extend="both", cmap=pyplot.cm.viridis)
extent = [t.min(), t.max(), 0, max(period)]
bx.contour(t, numpy.log2(period), sig95, [-99, 1], colors="k", linewidths=2, extent=extent)
bx.fill(numpy.concatenate([t, t[-1:] + dt, t[-1:] + dt, t[:1] - dt, t[:1] - dt]),
    numpy.concatenate([numpy.log2(coi), [1e-9],  numpy.log2(period[-1:]), numpy.log2(period[-1:]), [1e-9],]),"k",  alpha=0.3, hatch="x")
bx.set_title("b) {} Wavelet Power Spectrum ({})".format(label, mother.name))
bx.set_ylabel("Period (years)")
bx.hlines(numpy.log2(1), t[0], t[-1], colors='red', linestyles='dashed')

#
Yticks = 2 ** numpy.arange(numpy.ceil(numpy.log2(period.min())), numpy.ceil(numpy.log2(period.max())))
bx.set_yticks(numpy.log2(Yticks))
bx.set_yticklabels(Yticks)

# Third sub-plot, the global wavelet and Fourier power spectra and theoretical
# noise spectra. Note that period scale is logarithmic.
cx = pyplot.axes([0.77, 0.37, 0.2, 0.28], sharey=bx)
cx.plot(glbl_signif, numpy.log2(period), "k--")
cx.plot(var * fft_theor, numpy.log2(period), "--", color="#cccccc")
cx.plot( var * fft_power, numpy.log2(1.0 / fftfreqs), "-", color="#cccccc", linewidth=1.0)
cx.plot(var * glbl_power, numpy.log2(period), "k-", linewidth=1.5)
cx.set_title("c) Global Wavelet Spectrum")
cx.set_xlabel(r"Power [({})^2]".format(units))
cx.set_xlim([0, glbl_power.max() + var])
cx.set_ylim(numpy.log2([period.min(), period.max()]))
cx.set_yticks(numpy.log2(Yticks))
cx.set_yticklabels(Yticks)
pyplot.setp(cx.get_yticklabels(), visible=False)

# Fourth sub-plot, the scale averaged wavelet spectrum.
dx = pyplot.axes([0.1, 0.07, 0.65, 0.2], sharex=ax)
dx.axhline(scale_avg_signif, color="k", linestyle="--", linewidth=1.0)
dx.plot(t, scale_avg, "k-", linewidth=1.5)
dx.set_title("d) {}--{} year scale-averaged power".format(sel[0], sel[1]))
dx.set_xlabel("Time (year)")
dx.set_ylabel(r"Average variance [{}]".format(units))
ax.set_xlim([t.min(), t.max()])

pyplot.show()
#fig.savefig('wavelet'+label+'.pdf')

In [ ]:
dat = enso['ENSO'].values
par_name = 'enso'
title = "ENSO signal"
label = "ENSO Index"
units = ""
t0 = 1950
dt = 1/12  # In years

N = dat.size
t = numpy.arange(0, N) * dt + t0
locals()['t_'+par_name] = t


p = numpy.polyfit(t - t0, dat, 1)
dat_notrend = dat - numpy.polyval(p, t - t0)
std = dat_notrend.std()  # Standard deviation
var = std**2  # Variance
dat_norm = dat_notrend / std  # Normalized dataset
locals()['dat_norm_'+par_name] = dat_norm

mother = wavelet.Morlet(6)
s0 = 2 * dt  # Starting scale, in this case 2 * dt years = 2 months
dj = 1 / 12  # Twelve sub-octaves per octaves
J = 7 / dj  # Seven powers of two with dj sub-octaves
alpha, _, _ = wavelet.ar1(dat)  # Lag-1 autocorrelation for red noise

wave, scales, freqs, coi, fft, fftfreqs = wavelet.cwt(dat_norm, dt, dj, s0, J, mother)
iwave = wavelet.icwt(wave, scales, dt, dj, mother) * std

power = (numpy.abs(wave)) ** 2
fft_power = numpy.abs(fft) ** 2
period = 1 / freqs
locals()['period_'+par_name] = period

power /= scales[:, None]

In [ ]:
signif, fft_theor = wavelet.significance(
    1.0, dt, scales, 0, alpha, significance_level=0.95, wavelet=mother
)
sig95 = numpy.ones([1, N]) * signif[:, None]
sig95 = power / sig95
locals()['sig95_'+par_name] = sig95


glbl_power = power.mean(axis=1)
dof = N - scales  # Correction for padding at edges
glbl_signif, tmp = wavelet.significance(
    var, dt, scales, 1, alpha, significance_level=0.95, dof=dof, wavelet=mother
)

sel=np.array([1,2])
Cdelta = mother.cdelta
scale_avg = (scales * numpy.ones((N, 1))).transpose()
scale_avg = power / scale_avg  # As in Torrence and Compo (1998) equation 24
scale_avg = var * dj * dt / Cdelta * scale_avg[sel, :].sum(axis=0)
scale_avg_signif, tmp = wavelet.significance(
    var,
    dt,
    scales,
    2,
    alpha,
    significance_level=0.95,
    dof=[scales[sel[0]], scales[sel[-1]]],
    wavelet=mother,
)

In [ ]:
# Prepare the figure
pyplot.close("all")
pyplot.ioff()
figprops = dict(figsize=(11, 8), dpi=72)
fig = pyplot.figure(**figprops)

# First sub-plot, the original time series anomaly and inverse wavelet
# transform.
ax = pyplot.axes([0.1, 0.75, 0.65, 0.2])
ax.plot(t, iwave, "-", linewidth=1, color=[0.5, 0.5, 0.5])
ax.plot(t, dat, "k", linewidth=1.5)
ax.set_title("a) {}".format(title))
ax.set_ylabel(r"{} [{}]".format(label, units))

# Second sub-plot, the normalized wavelet power spectrum and significance
# level contour lines and cone of influece hatched area. Note that period
# scale is logarithmic.
bx = pyplot.axes([0.1, 0.37, 0.65, 0.28], sharex=ax)
levels = [0.0625, 0.125, 0.25, 0.5, 1, 2, 4, 8, 16]
bx.contourf(t, numpy.log2(period), numpy.log2(power), numpy.log2(levels), extend="both", cmap=pyplot.cm.viridis)
extent = [t.min(), t.max(), 0, max(period)]
bx.contour(t, numpy.log2(period), sig95, [-99, 1], colors="k", linewidths=2, extent=extent)
bx.fill(numpy.concatenate([t, t[-1:] + dt, t[-1:] + dt, t[:1] - dt, t[:1] - dt]),
    numpy.concatenate([numpy.log2(coi), [1e-9],  numpy.log2(period[-1:]), numpy.log2(period[-1:]), [1e-9],]),"k",  alpha=0.3, hatch="x")
bx.set_title("b) {} Wavelet Power Spectrum ({})".format(label, mother.name))
bx.set_ylabel("Period (years)")
bx.hlines(numpy.log2(1), t[0], t[-1], colors='red', linestyles='dashed')

#
Yticks = 2 ** numpy.arange(numpy.ceil(numpy.log2(period.min())), numpy.ceil(numpy.log2(period.max())))
bx.set_yticks(numpy.log2(Yticks))
bx.set_yticklabels(Yticks)

# Third sub-plot, the global wavelet and Fourier power spectra and theoretical
# noise spectra. Note that period scale is logarithmic.
cx = pyplot.axes([0.77, 0.37, 0.2, 0.28], sharey=bx)
cx.plot(glbl_signif, numpy.log2(period), "k--")
cx.plot(var * fft_theor, numpy.log2(period), "--", color="#cccccc")
cx.plot( var * fft_power, numpy.log2(1.0 / fftfreqs), "-", color="#cccccc", linewidth=1.0)
cx.plot(var * glbl_power, numpy.log2(period), "k-", linewidth=1.5)
cx.set_title("c) Global Wavelet Spectrum")
cx.set_xlabel(r"Power [({})^2]".format(units))
cx.set_xlim([0, glbl_power.max() + var])
cx.set_ylim(numpy.log2([period.min(), period.max()]))
cx.set_yticks(numpy.log2(Yticks))
cx.set_yticklabels(Yticks)
pyplot.setp(cx.get_yticklabels(), visible=False)

# Fourth sub-plot, the scale averaged wavelet spectrum.
dx = pyplot.axes([0.1, 0.07, 0.65, 0.2], sharex=ax)
dx.axhline(scale_avg_signif, color="k", linestyle="--", linewidth=1.0)
dx.plot(t, scale_avg, "k-", linewidth=1.5)
dx.set_title("d) {}--{} year scale-averaged power".format(sel[0], sel[1]))
dx.set_xlabel("Time (year)")
dx.set_ylabel(r"Average variance [{}]".format(units))
ax.set_xlim([t.min(), t.max()])

pyplot.show()
fig.savefig('wavelet'+label+'.pdf')

In [ ]:
dat = tide_resampled['Sea Level'].values - tide_resampled['Sea Level'].mean()
par_name = 'tide'
title = "Honoira Tide Gauge"
label = "Sea Level"
units = "mm"
t0 = 1968+7/12
dt = 1/12  # In years


N = dat.size
t = numpy.arange(0, N) * dt + t0
locals()['t_'+par_name] = t


p = numpy.polyfit(t - t0, dat, 1)
dat_notrend = dat - numpy.polyval(p, t - t0)
std = dat_notrend.std()  # Standard deviation
var = std**2  # Variance
dat_norm = dat_notrend / std  # Normalized dataset
locals()['dat_norm_'+par_name] = dat_norm

mother = wavelet.Morlet(6)
s0 = 2 * dt  # Starting scale, in this case 2 * dt years = 2 months
dj = 1 / 12  # Twelve sub-octaves per octaves
J = 7 / dj  # Seven powers of two with dj sub-octaves
alpha, _, _ = wavelet.ar1(dat)  # Lag-1 autocorrelation for red noise

wave, scales, freqs, coi, fft, fftfreqs = wavelet.cwt(dat_norm, dt, dj, s0, J, mother)
iwave = wavelet.icwt(wave, scales, dt, dj, mother) * std

power = (numpy.abs(wave)) ** 2
fft_power = numpy.abs(fft) ** 2
period = 1 / freqs
locals()['period_'+par_name] = period

power /= scales[:, None]

In [ ]:
signif, fft_theor = wavelet.significance(
    1.0, dt, scales, 0, alpha, significance_level=0.95, wavelet=mother
)
sig95 = numpy.ones([1, N]) * signif[:, None]
sig95 = power / sig95
locals()['sig95_'+par_name] = sig95


glbl_power = power.mean(axis=1)
dof = N - scales  # Correction for padding at edges
glbl_signif, tmp = wavelet.significance(
    var, dt, scales, 1, alpha, significance_level=0.95, dof=dof, wavelet=mother
)

sel=np.array([1,2])
Cdelta = mother.cdelta
scale_avg = (scales * numpy.ones((N, 1))).transpose()
scale_avg = power / scale_avg  # As in Torrence and Compo (1998) equation 24
scale_avg = var * dj * dt / Cdelta * scale_avg[sel, :].sum(axis=0)
scale_avg_signif, tmp = wavelet.significance(
    var,
    dt,
    scales,
    2,
    alpha,
    significance_level=0.95,
    dof=[scales[sel[0]], scales[sel[-1]]],
    wavelet=mother,
)

In [ ]:
# Prepare the figure
pyplot.close("all")
pyplot.ioff()
figprops = dict(figsize=(11, 8), dpi=72)
fig = pyplot.figure(**figprops)

# First sub-plot, the original time series anomaly and inverse wavelet
# transform.
ax = pyplot.axes([0.1, 0.75, 0.65, 0.2])
ax.plot(t, iwave, "-", linewidth=1, color=[0.5, 0.5, 0.5])
ax.plot(t, dat, "k", linewidth=1.5)
ax.set_title("a) {}".format(title))
ax.set_ylabel(r"{} [{}]".format(label, units))

# Second sub-plot, the normalized wavelet power spectrum and significance
# level contour lines and cone of influece hatched area. Note that period
# scale is logarithmic.
bx = pyplot.axes([0.1, 0.37, 0.65, 0.28], sharex=ax)
levels = [0.0625, 0.125, 0.25, 0.5, 1, 2, 4, 8, 16]
bx.contourf(t, numpy.log2(period), numpy.log2(power), numpy.log2(levels), extend="both", cmap=pyplot.cm.viridis)
extent = [t.min(), t.max(), 0, max(period)]
bx.contour(t, numpy.log2(period), sig95, [-99, 1], colors="k", linewidths=2, extent=extent)
bx.fill(numpy.concatenate([t, t[-1:] + dt, t[-1:] + dt, t[:1] - dt, t[:1] - dt]),
    numpy.concatenate([numpy.log2(coi), [1e-9],  numpy.log2(period[-1:]), numpy.log2(period[-1:]), [1e-9],]),"k",  alpha=0.3, hatch="x")
bx.set_title("b) {} Wavelet Power Spectrum ({})".format(label, mother.name))
bx.set_ylabel("Period (years)")
bx.hlines(numpy.log2(1), t[0], t[-1], colors='red', linestyles='dashed')

#
Yticks = 2 ** numpy.arange(numpy.ceil(numpy.log2(period.min())), numpy.ceil(numpy.log2(period.max())))
bx.set_yticks(numpy.log2(Yticks))
bx.set_yticklabels(Yticks)

# Third sub-plot, the global wavelet and Fourier power spectra and theoretical
# noise spectra. Note that period scale is logarithmic.
cx = pyplot.axes([0.77, 0.37, 0.2, 0.28], sharey=bx)
cx.plot(glbl_signif, numpy.log2(period), "k--")
cx.plot(var * fft_theor, numpy.log2(period), "--", color="#cccccc")
cx.plot( var * fft_power, numpy.log2(1.0 / fftfreqs), "-", color="#cccccc", linewidth=1.0)
cx.plot(var * glbl_power, numpy.log2(period), "k-", linewidth=1.5)
cx.set_title("c) Global Wavelet Spectrum")
cx.set_xlabel(r"Power [({})^2]".format(units))
cx.set_xlim([0, glbl_power.max() + var])
cx.set_ylim(numpy.log2([period.min(), period.max()]))
cx.set_yticks(numpy.log2(Yticks))
cx.set_yticklabels(Yticks)
pyplot.setp(cx.get_yticklabels(), visible=False)

# Fourth sub-plot, the scale averaged wavelet spectrum.
dx = pyplot.axes([0.1, 0.07, 0.65, 0.2], sharex=ax)
dx.axhline(scale_avg_signif, color="k", linestyle="--", linewidth=1.0)
dx.plot(t, scale_avg, "k-", linewidth=1.5)
dx.set_title("d) {}--{} year scale-averaged power".format(sel[0], sel[1]))
dx.set_xlabel("Time (year)")
dx.set_ylabel(r"Average variance [{}]".format(units))
ax.set_xlim([t.min(), t.max()])

pyplot.show()
fig.savefig('wavelet'+label+'.pdf')

In [ ]:
fred_resampled

In [ ]:
data_frame = fred_resampled['d13c_no_tectonic'].dropna()
dat = data_frame.values
#--------------------------------------
par_name = 'fred'
title = "d13C fred without tectonic signal"
label = "d13C"
units = ""
t0 = 1994.5
dt = 1/12  # In years

N = dat.size
t = numpy.arange(0, N) * dt + t0
locals()['t_'+par_name] = t

dat_notrend = signal.detrend(dat)
std = np.std(dat_notrend)
var = std ** 2  # Variance
dat_norm = dat_notrend / std  # Normalized dataset

mother = wavelet.Morlet(6)
s0 = 1 * dt  # Starting scale, in this case 1 * dt years = 2 months
dj = 1 / 24  # Twelve sub-octaves per octaves
J = 7 / dj  # Seven powers of two with dj sub-octaves
alpha, _, _ = wavelet.ar1(dat)  # Lag-1 autocorrelation for red noise

wave, scales, freqs, coi, fft, fftfreqs = wavelet.cwt(dat_norm, dt, dj, s0, J, mother)
iwave = wavelet.icwt(wave, scales, dt, dj, mother) * std

power = (np.abs(wave)) ** 2
fft_power = np.abs(fft) ** 2
period = 1 / freqs
locals()['period_'+par_name] = period

power /= scales[:, None]
signif, fft_theor = wavelet.significance(1.0, dt, scales, 0, alpha,
                                         significance_level=0.95,
                                         wavelet=mother)
sig95 = np.ones([1, N]) * signif[:, None]
sig95 = power / sig95
locals()['sig95_'+par_name] = sig95


glbl_power = power.mean(axis=1)
dof = N - scales  # Correction for padding at edges
glbl_signif, tmp = wavelet.significance(var, dt, scales, 1, alpha,
                                        significance_level=0.95, dof=dof,
                                        wavelet=mother)


sel = find((period >= 0.5) & (period < 4))
# We also calculate the scale average between 0.5 hours and 4 years, and its significance level.
Cdelta = mother.cdelta
scale_avg = (scales * np.ones((N, 1))).transpose()
scale_avg = power / scale_avg  # As in Torrence and Compo (1998) equation 24
scale_avg = var * dj * dt / Cdelta * scale_avg[sel, :].sum(axis=0)
scale_avg_signif, tmp = wavelet.significance(var, dt, scales, 2, alpha,
                                             significance_level=0.95,
                                             dof=[scales[sel[0]],
                                                  scales[sel[-1]]],
                                             wavelet=mother)

In [ ]:
# Prepare the figure
pyplot.close("all")
pyplot.ioff()
figprops = dict(figsize=(11, 8), dpi=72)
fig = pyplot.figure(**figprops)

# First sub-plot, the original time series anomaly and inverse wavelet
# transform.
ax = pyplot.axes([0.1, 0.75, 0.65, 0.2])
ax.plot(t, iwave, "-", linewidth=1, color=[0.5, 0.5, 0.5])
ax.plot(t, dat, "k", linewidth=1.5)
ax.set_title("a) {}".format(title))
ax.set_ylabel(r"{} [{}]".format(label, units))

# Second sub-plot, the normalized wavelet power spectrum and significance
# level contour lines and cone of influece hatched area. Note that period
# scale is logarithmic.
bx = pyplot.axes([0.1, 0.37, 0.65, 0.28], sharex=ax)
levels = [0.0625, 0.125, 0.25, 0.5, 1, 2, 4, 8, 16]
bx.contourf(t, numpy.log2(period), numpy.log2(power), numpy.log2(levels), extend="both", cmap=pyplot.cm.viridis)
extent = [t.min(), t.max(), 0, max(period)]
bx.contour(t, numpy.log2(period), sig95, [-99, 1], colors="k", linewidths=2, extent=extent)
bx.fill(numpy.concatenate([t, t[-1:] + dt, t[-1:] + dt, t[:1] - dt, t[:1] - dt]),
    numpy.concatenate([numpy.log2(coi), [1e-9],  numpy.log2(period[-1:]), numpy.log2(period[-1:]), [1e-9],]),"k",  alpha=0.3, hatch="x")
bx.set_title("b) {} Wavelet Power Spectrum ({})".format(label, mother.name))
bx.set_ylabel("Period (years)")
bx.hlines(numpy.log2(1), t[0], t[-1], colors='orange', linestyles='dashed')

#
Yticks = 2 ** numpy.arange(numpy.ceil(numpy.log2(period.min())), numpy.ceil(numpy.log2(period.max())))
bx.set_yticks(numpy.log2(Yticks))
bx.set_yticklabels(Yticks)

# Third sub-plot, the global wavelet and Fourier power spectra and theoretical
# noise spectra. Note that period scale is logarithmic.
cx = pyplot.axes([0.77, 0.37, 0.2, 0.28], sharey=bx)
cx.plot(glbl_signif, numpy.log2(period), "k--")
cx.plot(var * fft_theor, numpy.log2(period), "--", color="#cccccc")
cx.plot( var * fft_power, numpy.log2(1.0 / fftfreqs), "-", color="#cccccc", linewidth=1.0)
cx.plot(var * glbl_power, numpy.log2(period), "k-", linewidth=1.5)
cx.set_title("c) Global Wavelet Spectrum")
cx.set_xlabel(r"Power [({})^2]".format(units))
cx.set_xlim([0, glbl_power.max() + var])
cx.set_ylim(numpy.log2([period.min(), period.max()]))
cx.set_yticks(numpy.log2(Yticks))
cx.set_yticklabels(Yticks)
pyplot.setp(cx.get_yticklabels(), visible=False)

# Fourth sub-plot, the scale averaged wavelet spectrum.
dx = pyplot.axes([0.1, 0.07, 0.65, 0.2], sharex=ax)
dx.axhline(scale_avg_signif, color="k", linestyle="--", linewidth=1.0)
dx.plot(t, scale_avg, "k-", linewidth=1.5)
dx.set_title("d) {}--{} year scale-averaged power".format(sel[0], sel[1]))
dx.set_xlabel("Time (year)")
dx.set_ylabel(r"Average variance [{}]".format(units))
ax.set_xlim([t.min(), t.max()])

pyplot.show()
fig.savefig('wavelet'+label+'.pdf')

In [ ]:
data_frame = df_resampled.loc[:'2000','d13c_notec_ma'].dropna()
dat = data_frame.values
#--------------------------------------
par_name = 'd13c'
title = "d13C without tectonic signal"
label = "d13C"
units = ""
t0 = 1926.3490
dt = 1/12  # In years

N = dat.size
t = numpy.arange(0, N) * dt + t0
locals()['t_'+par_name] = t

dat_notrend = signal.detrend(dat)
std = np.std(dat_notrend)
var = std ** 2  # Variance
dat_norm = dat_notrend / std  # Normalized dataset

mother = wavelet.Morlet(6)
s0 = 1 * dt  # Starting scale, in this case 1 * dt years = 2 months
dj = 1 / 24  # Twelve sub-octaves per octaves
J = 7 / dj  # Seven powers of two with dj sub-octaves
alpha, _, _ = wavelet.ar1(dat)  # Lag-1 autocorrelation for red noise

wave, scales, freqs, coi, fft, fftfreqs = wavelet.cwt(dat_norm, dt, dj, s0, J, mother)
iwave = wavelet.icwt(wave, scales, dt, dj, mother) * std

power = (np.abs(wave)) ** 2
fft_power = np.abs(fft) ** 2
period = 1 / freqs
locals()['period_'+par_name] = period

power /= scales[:, None]
signif, fft_theor = wavelet.significance(1.0, dt, scales, 0, alpha,
                                         significance_level=0.95,
                                         wavelet=mother)
sig95 = np.ones([1, N]) * signif[:, None]
sig95 = power / sig95
locals()['sig95_'+par_name] = sig95



glbl_power = power.mean(axis=1)
dof = N - scales  # Correction for padding at edges
glbl_signif, tmp = wavelet.significance(var, dt, scales, 1, alpha,
                                        significance_level=0.95, dof=dof,
                                        wavelet=mother)


sel = find((period >= 0.5) & (period < 4))
# We also calculate the scale average between 0.5 hours and 4 years, and its significance level.
Cdelta = mother.cdelta
scale_avg = (scales * np.ones((N, 1))).transpose()
scale_avg = power / scale_avg  # As in Torrence and Compo (1998) equation 24
scale_avg = var * dj * dt / Cdelta * scale_avg[sel, :].sum(axis=0)
scale_avg_signif, tmp = wavelet.significance(var, dt, scales, 2, alpha,
                                             significance_level=0.95,
                                             dof=[scales[sel[0]],
                                                  scales[sel[-1]]],
                                             wavelet=mother)

In [ ]:
pyplot.close('all')
pyplot.ioff()

fig , axes = plt.subplots(2, figsize=(10,6), sharex=True, gridspec_kw={'hspace':0, 'height_ratios': [1,3]})

ax = axes[0]

ax.plot(t, dat_norm, 'k', linewidth=1.5, label='detrended data')
#ax.set_ylim(-0.01, 0.015)
ax.legend()
ax.grid()
ax.yaxis.tick_right()
ax.set_ylabel(r'$\delta^{13}$C')
ax.set_title(r'Coral $\delta^{13}$C' )

bx = axes[1]

levels = [0.0625, 0.125, 0.25, 0.5, 1, 2, 4, 8, 16]

bx.contourf(t, np.log2(period), np.log2(power), np.log2(levels),
            extend='both', cmap=pyplot.cm.viridis)
bx.contour(t, np.log2(period), sig95, [-99, 1], colors="k", linewidths=1, linestyles='dashed' , extent=extent)
bx.contour(t_enso, np.log2(period_enso), sig95_enso, [-99, 1], colors="r", linewidths=1, linestyles='dashed' , extent=extent)
bx.contour(t_tide, np.log2(period_tide), sig95_tide, [-99, 1], colors="orange", linewidths=1, linestyles='dashed' , extent=extent)


bx.set_ylim(np.log2(0.2),np.log2(6))
ytix = bx.get_yticks()
bx.set_yticks(ytix, labels=(2**ytix))
#bx.set_xticks(days)
#bx.set_xlim(np.datetime64('2022-08-15'))
#bx.tick_params(axis ='x', labelrotation =30)
plt.xticks(ha='right')
bx.grid(color='white',alpha=0.6)
bx.set_ylabel("Periods [years]")

#bx.hlines(23.934, days[0], days[-1], color='orange', linestyles='dashed', label='Principal lunar-solar')
#bx.hlines(11.967, days[0], days[-1], color='red', linestyles='dashed',    label='Lunar-solar')
bx.legend()
plt.show()


In [ ]:
pyplot.close('all')
pyplot.ioff()
fig, ax = plt.subplots(1, figsize=(15,5))

ax.contour( t_d13c, np.log2(period_d13c), sig95_d13c, [1,99], colors="k",linewidths=0.5, alpha=0.6, linestyles='solid', label=r'$\delta^{13}$C')
ax.contourf( t_d13c, np.log2(period_d13c), sig95_d13c, [1,99], colors="w", hatches=['\\\\'] )

ax.contour( t_enso, np.log2(period_enso), sig95_enso, [1,99], colors="r",linewidths=0.5,alpha=0.6,  linestyles='solid', label='ENSO')
cs = ax.contourf( t_enso, np.log2(period_enso), sig95_enso, [1,99], colors="w", hatches=['||'] )
cs.set_edgecolor("r") 

ax.contour( t_tide, np.log2(period_tide), sig95_tide, [1,99], colors="b",linewidths=0.5, alpha=0.6, linestyles='solid', label='Tide Gauge')
cs = ax.contourf(t_tide, np.log2(period_tide), sig95_tide, [1,99], colors="w", hatches=['xx']  )
cs.set_edgecolor("b") 

ax.set_ylim(np.log2(0.2),np.log2(6))
ytix = ax.get_yticks()
ax.set_yticks(ytix, labels=(2**ytix))
ax.set_ylabel("Periods [years]")
ax.grid()
plt.axhline(y=0, color='green', linestyle='dashed')

plt.show()

In [ ]:
pyplot.close('all')
pyplot.ioff()
fig, axes = plt.subplots(2,2, figsize=(12,6), sharex=True, gridspec_kw={'hspace':0, 'height_ratios':[2,1], 'width_ratios':[6,1]})

ax = axes[0,0]
ax.plot(df_sorted['age_absolute'], df_sorted['d13c'], ".", color='lightgreen', alpha=0.7, label=r'$\delta^{13}C$' )
ax.plot(df_sorted['age_absolute'], df_sorted['d13c_ma2'],  color='darkgreen', lw=2.5 )
#ax.plot(df_resampled['age_absolute'], df_resampled['d18o_ma2'],  color='orange', lw=2.5 )


ax.plot(fred['decimal_year'], fred['d13c'], ".", color='#40E0D0', alpha=0.3,  )
ax.plot(fred['decimal_year'], fred['d13c_ma'],  color='#1AC4A8', lw=0.75 )


#ax.vlines([1936+(31+29+22)/365, 1953-(25)/365, 2007+(31+29+31)/365],
#          ymin=20, ymax=30, color=['#F54927','#F5C827','#2D8708'])

ax.axvline(1936+(31+29+22)/365, color='#595959', label='M7.1 EQ 1936 Apr')
ax.axvline(2007+(31+29+31)/365, color='#595959', ls='--', label='M8.1 EQ 2007 Apr')



date1  = '1936-04'
date2  = '1941-06'
date2a = '1951-06'
date3  = '1966-02'
date4  = '1978-06'
date5  = '1987-06'
date6  = '2007-01'


                   ###########  1
x, y, p= line_fit(df_resampled.loc[:date1,'age_absolute'].values, 
               df_resampled.loc[:date1,'d13c'].values)
ax.plot(x, y, '-', color='#FF9724', linewidth=2.5, label=f'{p[0]:.3f}'+r'$\delta^{18}O$ ‰ per year')

                   ###########  2
x, y, p= line_fit(df_resampled.loc[date1:date2,'age_absolute'].values, 
               df_resampled.loc[date1:date2,'d13c'].values)
ax.plot(x, y, '-', color='#1316CF', linewidth=2.5, label=f'{p[0]:.3f}'+r'$\delta^{18}O$ ‰ per year')

x, y, p= line_fit(df_resampled.loc[date2:date2a,'age_absolute'].values, 
               df_resampled.loc[date2:date2a,'d13c'].values)
ax.plot(x, y, '--', color='#FF9724', linewidth=2.5, label=f'{p[0]:.3f}'+r'$\delta^{18}O$ ‰ per year')

                   ###########  3
x, y, p= line_fit(df_resampled.loc[date2a:date3,'age_absolute'].values, 
               df_resampled.loc[date2a:date3,'d13c'].values)
ax.plot(x, y, '-', color='#CF1352', linewidth=2.5, label=f'{p[0]:.3f}'+r'$\delta^{18}O$ ‰ per year')

                   ###########  4
x, y, p= line_fit(df_resampled.loc[date3:date4,'age_absolute'].values, 
               df_resampled.loc[date3:date4,'d13c'].values)
ax.plot(x, y, '--', color='#1316CF', linewidth=2.5, label=f'{p[0]:.3f}'+r'$\delta^{18}O$ ‰ per year')

                   ###########  5
x, y, p= line_fit(df_resampled.loc[date4:date5,'age_absolute'].values, 
               df_resampled.loc[date4:date5,'d13c'].values)
ax.plot(x, y, '--', color='#CF1352', linewidth=2.5, label=f'{p[0]:.3f}'+r'$\delta^{18}O$ ‰ per year')

                   ###########  6
x, y, p= line_fit(df_resampled.loc[date5:date6,'age_absolute'].values, 
               df_resampled.loc[date5:date6,'d13c'].values)
ax.plot(x, y, linestyle='dotted', color='#1316CF', linewidth=2.5, label=f'{p[0]:.3f}'+r'$\delta^{18}O$ ‰ per year')



ax.set_ylim(-2, 0)
ax.set_xlim(1927, 1960)

ax.grid()
#ax.legend(loc='lower left')

ax = axes[1,0]
ax.contour( t_d13c, np.log2(period_d13c), sig95_d13c, [1,99], colors="k",linewidths=0.5, alpha=0.6, linestyles='solid', label=r'$\delta^{13}$C')
ax.contourf( t_d13c, np.log2(period_d13c), sig95_d13c, [1,99], colors="w", hatches=['\\'] )
ax.contour( t_fred, np.log2(period_fred), sig95_fred, [1,99], colors="k",linewidths=0.5, alpha=0.6, linestyles='solid', label=r'$\delta^{13}$C')
ax.contourf( t_fred, np.log2(period_fred), sig95_fred, [1,99], colors="w", hatches=['//'] )

ax.contour( t_enso, np.log2(period_enso), sig95_enso, [1,99], colors="r",linewidths=0.5,alpha=0.6,  linestyles='solid', label='ENSO')
#cs = ax.contourf( t_enso, np.log2(period_enso), sig95_enso, [1,99], colors="w", hatches=['||'] )
cs.set_edgecolor("r") 

ax.contour( t_tide, np.log2(period_tide), sig95_tide, [1,99], colors="b",linewidths=0.5, alpha=0.6, linestyles='solid', label='Tide Gauge')
#cs = ax.contourf(t_tide, np.log2(period_tide), sig95_tide, [1,99], colors="w", hatches=['xx']  )
cs.set_edgecolor("b") 

ax.set_ylim(np.log2(0.2),np.log2(4))
ytix = ax.get_yticks()
ax.set_yticks(ytix, labels=(2**ytix))
ax.set_ylabel("Periods [years]")
ax.grid()
ax.axhline(y=0, color='green', linestyle='dashed')
ax.axvline(1936+(31+29+22)/365, color='#595959', label='M7.1 EQ 1936 Apr')
ax.axvline(2007+(31+29+31)/365, color='#595959', ls='--', label='M8.1 EQ 2007 Apr')

legend_handles = [
    Patch(facecolor='white', edgecolor='k', hatch='\\\\', label=r'$\delta^{13}$C'),
    Patch(facecolor='white', edgecolor='k', hatch='////', label=r'$\delta^{13}$C fred'),
    Patch(facecolor='white', edgecolor='r', hatch='||', label='ENSO'),
    Patch(facecolor='white', edgecolor='b', hatch='xx', label='Tide Gauge'),
    Line2D([0], [0], color='#595959', linewidth=2, linestyle='-', label='M7.1 EQ 1936 Apr'),
    Line2D([0], [0], color='#595959', linewidth=2, linestyle='--', label='M8.1 EQ 2007 Apr')]

# Add legend to your axis
#ax.legend(bbox_to_anchor=(1.25, 1), handles=legend_handles, loc='upper right', title='Contours')
axes[1,0].legend(
    handles=legend_handles,
    loc='upper left',
    bbox_to_anchor=(1.02, 1),
    borderaxespad=0.,
    frameon=False,
    title='Contours'
)

axes[0,0].legend(
    loc='upper left',
    bbox_to_anchor=(1.02, 1),  # just to the right of the upper left corner
    borderaxespad=0.,
    frameon=False,
    title='Trends'
)

axes[1,1].axis('off')
axes[0,1].axis('off')
plt.show()
#fig.savefig('wavelet_1925-1950.pdf')

In [ ]:
d = [t_enso, np.log2(period_enso), sig95_enso]
for i in range(3):
    dx = d[i]
    print(dx.shape)

In [ ]:
pyplot.close('all')
pyplot.ioff()
fig, ax = plt.subplots(1,figsize=(8,4))
ax.imshow(sig95_enso)
plt.show()